# Customer Churn Analysis

### Python | Pandas | NumPy | Data Visualization

**Project Objective:** Analyze customer churn behavior, identify important churn patterns, and prepare a clean dataset for SQL and Power BI analysis.

**Business Goal:** Understand which customer segments are more likely to churn and generate actionable insights for customer retention.


# 1. Project Introduction

This project analyzes customer churn using the Telco Customer Churn dataset.

### Objectives
- Understand the structure and quality of customer data
- Clean and preprocess the dataset
- Analyze churn using Pandas and NumPy
- Create useful customer segments
- Visualize major churn patterns
- Prepare a cleaned dataset for SQL and Power BI

### Tools Used
- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Google Colab


# 2. Import Libraries

Import the Python libraries required for data cleaning, analysis, numerical calculations, and visualization.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

# Visualization settings
sns.set_theme(style="whitegrid")


# 3. Load Dataset

Upload **Telco_customer_churn.xlsx** using the Colab **Files → Upload** option before running this cell.

The dataset is loaded into a Pandas DataFrame for analysis.


In [ ]:
# Load the uploaded Excel dataset
df = pd.read_excel("/content/Telco_customer_churn.xlsx")

# Preview the data
df.head()


# 4. Data Understanding

Before cleaning the data, inspect its structure, dimensions, columns, data types, summary statistics, missing values, and duplicate records.


In [ ]:
# First 5 rows
df.head()


In [ ]:
# Number of rows and columns
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


In [ ]:
# Column names
df.columns.tolist()


In [ ]:
# Data types and non-null values
df.info()


In [ ]:
# Numerical summary
df.describe(include="all").T


In [ ]:
# Missing values in each column
df.isnull().sum().sort_values(ascending=False)


In [ ]:
# Duplicate rows
print("Duplicate rows:", df.duplicated().sum())


# 5. Data Cleaning

The dataset is cleaned by:
- Removing duplicate rows
- Converting `Total Charges` into a numeric column
- Handling missing values created during conversion
- Standardizing column names

The raw dataset is not overwritten; cleaning is performed in the DataFrame used for analysis.


In [ ]:
# Remove duplicate rows
df = df.drop_duplicates().copy()

print("Duplicate rows after cleaning:", df.duplicated().sum())


In [ ]:
# Check data types before conversion
df.dtypes


In [ ]:
# Convert Total Charges to numeric
df["Total Charges"] = pd.to_numeric(
    df["Total Charges"],
    errors="coerce"
)

print("Missing Total Charges after conversion:",
      df["Total Charges"].isnull().sum())


In [ ]:
# Inspect rows where Total Charges could not be converted
df[df["Total Charges"].isnull()].head()


In [ ]:
# Fill missing Total Charges values
df["Total Charges"] = df["Total Charges"].fillna(0)


In [ ]:
# Standardize column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df.columns.tolist()


# 6. Pandas Analysis

Pandas is used to calculate customer counts, churn rate, and churn patterns across important customer attributes.


In [ ]:
# Total unique customers
total_customers = df["customerid"].nunique()
print("Total Customers:", total_customers)


In [ ]:
# Churn distribution
df["churn_label"].value_counts()


In [ ]:
# Churn percentage
churn_percentage = (
    df["churn_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

churn_percentage


In [ ]:
# Number of churned customers
churned_customers = df.loc[
    df["churn_label"].eq("Yes"),
    "customerid"
].nunique()

print("Churned Customers:", churned_customers)


In [ ]:
# Overall churn rate
churn_rate = df["churn_label"].eq("Yes").mean() * 100
print(f"Overall Churn Rate: {churn_rate:.2f}%")


In [ ]:
# Churn by contract
pd.crosstab(df["contract"], df["churn_label"])


In [ ]:
# Churn by internet service
pd.crosstab(df["internet_service"], df["churn_label"])


In [ ]:
# Churn by payment method
pd.crosstab(df["payment_method"], df["churn_label"])


In [ ]:
# Average monthly charges by churn status
df.groupby("churn_label")["monthly_charges"].mean().round(2)


In [ ]:
# Average tenure by churn status
df.groupby("churn_label")["tenure_months"].mean().round(2)


# 7. NumPy Analysis

NumPy is used for numerical and statistical analysis of monthly customer charges.


In [ ]:
# Convert monthly charges to a NumPy array
monthly_charges = np.array(df["monthly_charges"].dropna())

print("Mean:", np.mean(monthly_charges))
print("Median:", np.median(monthly_charges))
print("Maximum:", np.max(monthly_charges))
print("Minimum:", np.min(monthly_charges))
print("Standard Deviation:", np.std(monthly_charges))


In [ ]:
# NumPy analysis for churned customers
churned_charges = np.array(
    df.loc[df["churn_label"].eq("Yes"), "monthly_charges"].dropna()
)

print("Churned Customer Average:", np.mean(churned_charges))
print("Churned Customer Median:", np.median(churned_charges))
print("Churned Customer Maximum:", np.max(churned_charges))
print("Churned Customer Minimum:", np.min(churned_charges))


# 8. Feature Engineering

Create a `tenure_group` feature to segment customers based on how long they have stayed with the company.


In [ ]:
# Create tenure groups
df["tenure_group"] = pd.cut(
    df["tenure_months"],
    bins=[-1, 12, 24, 48, 72],
    labels=[
        "0-12 Months",
        "13-24 Months",
        "25-48 Months",
        "49-72 Months"
    ]
)

df["tenure_group"].value_counts().sort_index()


In [ ]:
# Churn rate by tenure group
churn_by_tenure = (
    df.groupby("tenure_group", observed=True)["churn_label"]
    .apply(lambda x: x.eq("Yes").mean() * 100)
    .round(2)
    .reset_index(name="churn_rate")
)

churn_by_tenure


# 9. Data Visualization

Visualizations help identify the main customer churn patterns and relationships between customer attributes and churn.


In [ ]:
# 1. Customer Churn Distribution
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="churn_label")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn Status")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()


In [ ]:
# 2. Churn by Contract Type
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="contract", hue="churn_label")
plt.title("Customer Churn by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Number of Customers")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


In [ ]:
# 3. Monthly Charges vs Churn
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="churn_label", y="monthly_charges")
plt.title("Monthly Charges vs Churn")
plt.xlabel("Churn Status")
plt.ylabel("Monthly Charges")
plt.tight_layout()
plt.show()


In [ ]:
# 4. Tenure vs Churn
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="churn_label", y="tenure_months")
plt.title("Tenure vs Churn")
plt.xlabel("Churn Status")
plt.ylabel("Tenure (Months)")
plt.tight_layout()
plt.show()


In [ ]:
# 5. Churn Rate by Tenure Group
plt.figure(figsize=(9, 5))
sns.barplot(data=churn_by_tenure, x="tenure_group", y="churn_rate")
plt.title("Churn Rate by Tenure Group")
plt.xlabel("Tenure Group")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


# 10. Key Findings

Use the analysis results above to record the most important business insights.

### Findings to Review
- Compare churn across contract types.
- Identify tenure groups with higher churn rates.
- Compare monthly charges between churned and retained customers.
- Review internet service and payment method patterns.
- Identify customer segments that may require stronger retention strategies.

> **Note:** Final findings should be based on the actual outputs generated from this notebook rather than assumptions.


# 11. Final Data Quality Check

Perform a final check before exporting the cleaned dataset.


In [ ]:
print("Dataset Shape:", df.shape)
print("\nMissing Values:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

print("\nDuplicate Rows:", df.duplicated().sum())


# 12. Export Clean Dataset

The cleaned dataset is exported as a CSV file.

This file can be used as the input for the next stages of the project:
**SQL Analysis → Power BI Dashboard → GitHub Documentation**.


In [ ]:
# Export cleaned dataset
output_file = "customer_churn_cleaned.csv"

df.to_csv(output_file, index=False)

print(f"Cleaned dataset exported successfully: {output_file}")


# Project Workflow

**Python / Google Colab**
→ Data Cleaning & Analysis
→ **SQL**
→ Business Queries
→ **Power BI**
→ 2-Page Dashboard
→ **GitHub**
→ Project Documentation

### Project Complete
The Python analysis stage is now ready for SQL and Power BI.
